# `aridkpi` tutorial — KPIs de resiliencia para climas áridos

Este notebook recorre el flujo completo de uso del paquete `aridkpi` con datos
sintéticos que emulan una vivienda en clima BWk (Mendoza, Argentina). Al final
vas a saber:

1. Cómo generar datos sintéticos para distintas tipologías constructivas.
2. Cómo cargar datos reales de monitoreo (HOBO CSV) o simulación (EnergyPlus).
3. Cómo calcular los 5 KPIs CORE.
4. Cómo visualizar y comparar tipologías.
5. Cómo interpretar los resultados.

**Tiempo estimado:** 10 min

## 1. Setup

In [ ]:
import aridkpi
import pandas as pd
import matplotlib.pyplot as plt

print(f'aridkpi version: {aridkpi.__version__}')

## 2. Generar datos sintéticos

Empezamos con datos sintéticos para evitar dependencias externas.
El paquete provee un generador parametrizado por tipología constructiva: adobe,
mampostería sin aislación, mampostería con aislación según IRAM 11605, y sistema
industrializado liviano.

Los parámetros internos (atenuación térmica, desfase) emulan las propiedades
documentadas en la literatura para el clima de Cuyo.

In [ ]:
df = aridkpi.synth.synthetic_dataset(
    start='2026-01-15',
    days=14,
    typology='masonry_insulated',
    seed=42,
)
df.head()

In [ ]:
df.describe()

Notar la amplitud térmica diaria del exterior (típica de BWk: rango de unos 16 °C)
y la atenuación térmica que produce la envolvente sobre el interior.

In [ ]:
ax = aridkpi.viz.plot_temperature_series(
    df['T_in'], df['T_ext'],
    title='Vivienda con mampostería + aislación (IRAM 11605) — Mendoza, 2 semanas de enero',
)
plt.tight_layout()
plt.show()

## 3. Calcular los 5 KPIs CORE

### 3.1. IOD — Indoor Overheating Degree

Magnitud de sobrecalentamiento. Umbral de confort 26 °C (categoría II de
EN 16798-1).

In [ ]:
iod_value = aridkpi.iod(df['T_in'], T_comf=26.0)
print(f'IOD = {iod_value:.2f} °C·h')

In [ ]:
ax = aridkpi.viz.plot_overheating_diagnostic(
    df['T_in'], T_comf=26.0,
    title=f'Diagnóstico visual de IOD = {iod_value:.1f} °C·h',
)
plt.tight_layout()
plt.show()

El área sombreada bajo la curva por encima del umbral es exactamente
la integral del IOD.

### 3.2. UDH — Unmet Degree Hours durante un corte de luz

Simulamos un corte de 72 h durante el pico de calor. La habitabilidad pasiva
(Passive Survivability) se mide por las horas que el interior supera 30 °C
durante el corte.

In [ ]:
outage_start = df['T_ext'].idxmax()
print(f'Pico de temperatura exterior: {outage_start} ({df["T_ext"].max():.1f} °C)')

# El corte arranca al inicio del día del pico
outage_start = outage_start.normalize()

udh_24 = aridkpi.udh(df['T_in'], outage_start=outage_start, window='24h', threshold=30.0)
udh_72 = aridkpi.udh(df['T_in'], outage_start=outage_start, window='72h', threshold=30.0)

print(f'UDH 24h: {udh_24:.2f} °C·h')
print(f'UDH 72h: {udh_72:.2f} °C·h')

### 3.3. dT/dt máx — tasa máxima de cambio térmico

Velocidad máxima con la que cambia la temperatura interior. Discrimina muy bien
entre tipologías de alta y baja inercia térmica.

In [ ]:
rate = aridkpi.max_thermal_change_rate(df['T_in'])
print(f'dT/dt máx = {rate:.3f} °C/h')

### 3.4. CCOR — Climate Change Overheating Resistivity

Para CCOR necesitamos comparar contra una baseline. Simulamos una vivienda
lightweight (no resiliente) como baseline y la actual como estrategia.
Asumimos un calentamiento climático de 2.5 °C bajo SSP2-4.5/2080.

In [ ]:
df_baseline = aridkpi.synth.synthetic_dataset(
    start='2026-01-15', days=14, typology='lightweight', seed=42,
)
iod_baseline = aridkpi.iod(df_baseline['T_in'], T_comf=26.0)
iod_strategy = aridkpi.iod(df['T_in'], T_comf=26.0)

ccor_value = aridkpi.ccor(
    iod_baseline=iod_baseline,
    iod_strategy=iod_strategy,
    delta_T_climate=2.5,
)
print(f'IOD baseline (lightweight)         : {iod_baseline:.2f} °C·h')
print(f'IOD strategy  (masonry + insulation): {iod_strategy:.2f} °C·h')
print(f'CCOR                                : {ccor_value:.2f} °C·h/°C')

### 3.5. ΔE/ΔT — sensibilidad energética al clima

Para ΔE/ΔT se necesitan datos agregados anuales bajo varios SSP × horizonte.
Simulamos los resultados de 4 escenarios de simulación EnergyPlus.

In [ ]:
ssp_results = pd.DataFrame({
    'scenario': ['Baseline', 'SSP1-2.6/2050', 'SSP2-4.5/2050', 'SSP2-4.5/2080', 'SSP5-8.5/2080'],
    'T_mean':   [16.5, 17.4, 18.2, 19.1, 21.0],
    'EUI':      [82.0, 86.5, 92.0, 98.0, 110.0],
})
result = aridkpi.energy_climate_sensitivity(ssp_results)
print(f"ΔE/ΔT  (slope)     : {result['slope']:.2f} kWh·m⁻²·yr⁻¹/°C")
print(f"R²                 : {result['r_squared']:.4f}")
print(f"p-value            : {result['p_value']:.4g}")
print(f"Standard error     : {result['std_err']:.3f}")

## 4. Comparación entre tipologías

Comparamos los 4 tipos de envolvente sobre el mismo clima sintético.

In [ ]:
typologies = ['adobe', 'masonry_no_insulation', 'masonry_insulated', 'lightweight']
results = {}
for typology in typologies:
    d = aridkpi.synth.synthetic_dataset(start='2026-01-15', days=14, typology=typology, seed=42)
    results[typology] = {
        'IOD':       aridkpi.iod(d['T_in'], T_comf=26.0),
        'dT/dt_max': aridkpi.max_thermal_change_rate(d['T_in']),
        'UDH_72h':   aridkpi.udh(d['T_in'],
                                 outage_start=d['T_ext'].idxmax().normalize(),
                                 window='72h', threshold=30.0),
    }

summary = pd.DataFrame(results).T
summary.style.format(precision=2).background_gradient(cmap='RdYlGn_r')

In [ ]:
ax = aridkpi.viz.plot_kpi_summary(
    results,
    title='Comparación de tipologías constructivas — Mendoza enero, 2 semanas',
)
plt.tight_layout()
plt.show()

## 5. Interpretación

Los resultados muestran el ordenamiento esperado:

* **Adobe** (alta inercia, bajo dT/dt máx) presenta el mejor desempeño en
  estabilidad dinámica y el menor IOD.
* **Lightweight** (baja inercia, alto dT/dt máx) es el más vulnerable: sigue
  rápidamente al exterior, acumula más sobrecalentamiento y pierde
  habitabilidad rápido durante un corte.
* **Mampostería con aislación IRAM 11605** equilibra ambos efectos.

Estos resultados son ilustrativos —datos sintéticos. Los KPIs cobran su valor
real cuando se aplican a casos monitoreados y modelos calibrados
bayesianamente, como se prevé en el plan 2026–2029.

## 6. Próximos pasos

Si querés profundizar:

* La documentación completa está en https://gbarea-INAHE.github.io/aridkpi/
* Cada KPI tiene su entrada en la matriz comparativa
  (https://doi.org/10.5281/zenodo.19964373) con fórmula, supuestos,
  limitaciones BWk/BSk y referencias bibliográficas.
* Para cargar tus propios datos: `aridkpi.io.load_hobo_csv()`,
  `load_generic_csv()`, `load_energyplus_csv()`.